In [121]:
import pandas as pd

# Load the CSV files
matches_file_path = 'data/epic7_matches.csv'
hero_codes_file_path = 'data/hero_code_to_name.csv'

matches_df = pd.read_csv(matches_file_path)

matches_df.head()

,my_team_picks,enemy_team_picks,my_bans,enemy_bans,first_pick_team,winner
0,"[{""hero_code"": ""c2011"", ""pick_order"": 1}, {""he...","[{""hero_code"": ""c2007"", ""pick_order"": 2}, {""he...","[""c4158"", ""c2066""]","[""c1133"", ""c2066""]",my_team,my_team
1,"[{""hero_code"": ""c2011"", ""pick_order"": 1}, {""he...","[{""hero_code"": ""c2008"", ""pick_order"": 2}, {""he...","[""c2066"", """"]","[""c2113"", """"]",my_team,my_team
2,"[{""hero_code"": ""c5110"", ""pick_order"": 2}, {""he...","[{""hero_code"": ""c2011"", ""pick_order"": 1}, {""he...","[""c2066"", """"]","[""c2090"", """"]",enemy_team,enemy_team
3,"[{""hero_code"": ""c2011"", ""pick_order"": 1}, {""he...","[{""hero_code"": ""c2106"", ""pick_order"": 2}, {""he...","[""c4158"", ""c2066""]","[""c2066"", ""c2113""]",my_team,my_team
4,"[{""hero_code"": ""c5110"", ""pick_order"": 2}, {""he...","[{""hero_code"": ""c2008"", ""pick_order"": 1}, {""he...","[""c4158"", ""c2066""]","[""c2011"", ""c1153""]",enemy_team,my_team


In [122]:
# Iterate over each match and apply the rules
lst = []
for index, row in matches_df.iterrows():
    winner = row['winner']  # Adjust this if your column name is different
    
    # Map winner to my_team and loser to enemy_team
    first_pick_mapping = {'my_team':'winning_team', 'enemy_team':'losing_team'} if winner == 'my_team' else {'my_team':'losing_team', 'enemy_team':'winning_team'}
    mapping = {'winning_team':'my', 'losing_team':'enemy'} if winner == 'my_team' else {'winning_team':'enemy', 'losing_team':'my'}

    new_row = {
        'winning_team_picks': row[mapping['winning_team'] + '_team_picks'],
        'losing_team_picks': row[mapping['losing_team'] + '_team_picks'],
        'winning_team_bans': row[mapping['winning_team'] + '_bans'],
        'losing_team_bans': row[mapping['losing_team'] + '_bans'],
        'first_pick_team': first_pick_mapping[row['first_pick_team']] 
    }

    # Append the new row
    lst.append(new_row)

    
# Initialize a new DataFrame to store the rearranged data
rearranged_matches_df = pd.DataFrame(lst, columns=[
    'winning_team_picks',
    'losing_team_picks',
    'winning_team_bans',
    'losing_team_bans',
    'first_pick_team'
])

# Shuffle the matches
rearranged_matches_df = rearranged_matches_df.sample(frac=1).reset_index(drop=True)

# Save the new DataFrame to a CSV file
rearranged_matches_df.to_csv('data/rearranged_epic7_matches.csv', index=False)
print('Rearranged data has been saved to data/rearranged_epic7_matches.csv')

rearranged_matches_df.head()

Rearranged data has been saved to data/rearranged_epic7_matches.csv


,winning_team_picks,losing_team_picks,winning_team_bans,losing_team_bans,first_pick_team
0,"[{""hero_code"": ""c2011"", ""pick_order"": 1}, {""he...","[{""hero_code"": ""c1117"", ""pick_order"": 2}, {""he...","[""c2112"", """"]","[""c3002"", """"]",winning_team
1,"[{""hero_code"": ""c2011"", ""pick_order"": 2}, {""he...","[{""hero_code"": ""c2016"", ""pick_order"": 1}, {""he...","[""c1133"", """"]","[""c2066"", """"]",losing_team
2,"[{""hero_code"": ""c2090"", ""pick_order"": 2}, {""he...","[{""hero_code"": ""c1153"", ""pick_order"": 1}, {""he...","[""c2011"", """"]","[""c2011"", """"]",losing_team
3,"[{""hero_code"": ""c2090"", ""pick_order"": 2}, {""he...","[{""hero_code"": ""c1055"", ""pick_order"": 1}, {""he...","[""c2066"", ""c5110""]","[""c2113"", ""c2011""]",losing_team
4,"[{""hero_code"": ""c5110"", ""pick_order"": 2}, {""he...","[{""hero_code"": ""c2036"", ""pick_order"": 1}, {""he...","[""c2066"", ""c4158""]","[""c2011"", ""c2112""]",losing_team


In [123]:
import numpy as np
import ast
import pandas as pd

# Load the hero codes mapping
hero_mapping_df = pd.read_csv(hero_codes_file_path, header=None, names=['code', 'name'])
hero_codes = hero_mapping_df['code'].tolist()

# Define the vocabulary
vocab = ['Null', 'Start', 'Ban', 'Pick', 'winning_team', 'losing_team', 'c_null'] + hero_codes + [str(i) for i in range(11)]
vocab_size = len(vocab)

# Create a mapping from word to index for one-hot encoding
word_to_index = {word: idx for idx, word in enumerate(vocab)}
index_to_word = {idx: word for word, idx in word_to_index.items()}

# Prepare a function to convert JSON-like strings into lists of dictionaries
def parse_json_like_string(json_str):
    try:
        return ast.literal_eval(json_str)
    except ValueError:
        return []

# Prepare a function to convert a match into a sequence
def draft_to_sequence(match):
    sequence = []
    
    # Parse the JSON-like strings
    winning_picks = parse_json_like_string(match['winning_team_picks'])
    losing_picks = parse_json_like_string(match['losing_team_picks'])
    winning_bans = parse_json_like_string(match['winning_team_bans'])
    losing_bans = parse_json_like_string(match['losing_team_bans'])
        
    # Add the bans to the sequence (pick order 0)
    for ban in winning_bans:
        sequence.extend(['Ban', '0', 'winning_team', ban if ban else 'c_null'])
    
    for ban in losing_bans:
        sequence.extend(['Ban', '0', 'losing_team', ban if ban else 'c_null'])
    
    # Combine and sort all picks to ensure order correctness
    all_picks = winning_picks + losing_picks
    all_picks_sorted = sorted(all_picks, key=lambda x: x['pick_order'])
    
    # Add picks to the sequence in sorted order
    for pick in all_picks_sorted:
        team = 'winning_team' if pick in winning_picks else 'losing_team'
        sequence.extend(['Pick', str(pick['pick_order']), team, pick['hero_code']])
    
    # Pad the sequence with 'Null' words to reach 56 tokens
    while len(sequence) < 56:
        sequence.append('Null')

    return sequence

# Convert all matches into sequences
sequences = rearranged_matches_df.apply(draft_to_sequence, axis=1).tolist()

# Function to add padding between sequences and initial padding
def add_padding_between_sequences(sequences):
    padded_sequences = []
    padding = ['Null'] * 55 + ['Start']
    
    for sequence in sequences:
        padded_sequences.extend(padding)
        padded_sequences.extend(sequence)

    return padded_sequences

# Add padding between the sequences
padded_sequence = add_padding_between_sequences(sequences)

# Convert the padded sequence into one-hot encoded format
def one_hot_encode_sequence(sequence, vocab_size, word_to_index):
    one_hot_encoded = np.zeros((len(sequence), vocab_size), dtype=np.int32)
    for i, word in enumerate(sequence):
        word_index = word_to_index.get(word, word_to_index['Null'])
        one_hot_encoded[i, word_index] = 1
    return one_hot_encoded

# One-hot encode the entire padded sequence
one_hot_encoded_sequence = one_hot_encode_sequence(padded_sequence, vocab_size, word_to_index)

# Output the shape of the encoded data for verification
print(f'One-hot encoded sequence shape: {one_hot_encoded_sequence.shape}')


One-hot encoded sequence shape: (5600, 357)


In [124]:
decoded_sequence = [index_to_word[np.argmax(token)] for token in one_hot_encoded_sequence]
decoded_sentence = ' '.join(decoded_sequence)

# Split the decoded sentence into lines with 4 words each
words = decoded_sentence.split()
lines = [' '.join(words[i:i+4]) for i in range(0, len(words), 4)]
formatted_sentence = '\n'.join(lines)

print(formatted_sentence)


Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Start
Ban 0 winning_team c2112
Ban 0 winning_team c_null
Ban 0 losing_team c3002
Ban 0 losing_team c_null
Pick 1 winning_team c2011
Pick 2 losing_team c1117
Pick 3 losing_team c2002
Pick 4 winning_team c2090
Pick 5 winning_team c1153
Pick 6 losing_team c1133
Pick 7 losing_team c1103
Pick 8 winning_team c2007
Pick 9 winning_team c1089
Pick 10 losing_team c1118
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Start
Ban 0 winning_team c1133
Ban 0 winning_team c_null
Ban 0 losing_team c2066
Ban 0 l